In [13]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotnine as p9
import geopandas as gpd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import plotly.io as pio
pio.renderers.default = "notebook"

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time


pd.set_option('display.max_columns', None)


In [2]:
file_paths = ["WorldCupMatches.csv", "WorldCupPlayers.csv", "WorldCups.csv"]
df_list = []
for i in file_paths:
# Set the path to the file you'd like to load
    file_path = i

    # Load the latest version
    df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "abecklas/fifa-world-cup",
    file_path,
    # Provide any additional arguments like 
    # sql_query or pandas_kwargs. See the 
    # documenation for more information:
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
    )
    df_list.append(df)

matches_df = df_list[0]
players_df = df_list[1]
cups_df = df_list[2]

print(f'cups_df.columns: {cups_df.columns}')    
print(f'matches_df.columns: {matches_df.columns}')
print(f'players_df.columns: {players_df.columns}')

/var/folders/60/m3wm021x08vbp7kk3s0t94c40000gn/T/ipykernel_61270/2552925388.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.


cups_df.columns: Index(['Year', 'Country', 'Winner', 'Runners-Up', 'Third', 'Fourth',
       'GoalsScored', 'QualifiedTeams', 'MatchesPlayed', 'Attendance'],
      dtype='str')
matches_df.columns: Index(['Year', 'Datetime', 'Stage', 'Stadium', 'City', 'Home Team Name',
       'Home Team Goals', 'Away Team Goals', 'Away Team Name',
       'Win conditions', 'Attendance', 'Half-time Home Goals',
       'Half-time Away Goals', 'Referee', 'Assistant 1', 'Assistant 2',
       'RoundID', 'MatchID', 'Home Team Initials', 'Away Team Initials'],
      dtype='str')
players_df.columns: Index(['RoundID', 'MatchID', 'Team Initials', 'Coach Name', 'Line-up',
       'Shirt Number', 'Player Name', 'Position', 'Event'],
      dtype='str')


In [16]:
# Home / away win rates by stadium
m = matches_df.copy()
m['Stadium'] = m['Stadium'].astype(str).str.strip()
m['City'] = m['City'].astype(str).str.strip()
m['Home Team Goals'] = pd.to_numeric(m['Home Team Goals'], errors='coerce')
m['Away Team Goals'] = pd.to_numeric(m['Away Team Goals'], errors='coerce')
m = m.dropna(subset=['Stadium', 'Home Team Goals', 'Away Team Goals'])
m = m[m['Stadium'].ne('') & m['Stadium'].ne('nan')]

m['home_win'] = (m['Home Team Goals'] > m['Away Team Goals']).astype(int)
m['away_win'] = (m['Home Team Goals'] < m['Away Team Goals']).astype(int)
m['draw'] = (m['Home Team Goals'] == m['Away Team Goals']).astype(int)

stadium_rates = (
    m.groupby('Stadium', as_index=False)
     .agg(
         City=('City', 'first'),
         matches=('home_win', 'size'),
         home_win_rate=('home_win', 'mean'),
         away_win_rate=('away_win', 'mean'),
         draw_rate=('draw', 'mean'),
     )
     .sort_values('matches', ascending=False)
)

stadium_rates['home_win_rate'] *= 100
stadium_rates['away_win_rate'] *= 100
stadium_rates['draw_rate'] *= 100

print(f'Stadiums: {len(stadium_rates)}')
print(f'Overall home win rate: {m["home_win"].mean():.1%}')
print(f'Overall away win rate: {m["away_win"].mean():.1%}')
stadium_rates.head(20)

Stadiums: 181
Overall home win rate: 57.3%
Overall away win rate: 20.4%


,Stadium,City,matches,home_win_rate,away_win_rate,draw_rate
32,Estadio Azteca,Mexico City,19,73.684211,10.526316,15.789474
78,Jalisco,Guadalajara,14,71.428571,21.428571,7.142857
116,Olympiastadion,Berlin West,14,42.857143,28.571429,28.571429
110,Nou Camp - Estadio Le�n,Leon,11,54.545455,27.272727,18.181818
48,Estadio do Maracana,Rio De Janeiro,10,60.000000,30.000000,10.000000
104,Nacional,Santiago De Chile,10,90.000000,0.000000,10.000000
35,Estadio Centenario,Montevideo,10,100.000000,0.000000,0.000000
42,Estadio Nacional,Brasilia,10,70.000000,30.000000,0.000000
157,Stade de France,Saint-Denis,9,44.444444,22.222222,33.333333
160,Stade du Parc Lescure,Bordeaux,9,33.333333,11.111111,55.555556


In [18]:
# Geocode stadiums (Stadium + City) with Nominatim; cache within session

geolocator = Nominatim(user_agent='fifa_world_cup_stadium_maps')
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

if 'geocode_cache' not in globals():
    geocode_cache = {}

def lookup_coords(stadium, city):
    key = (str(stadium).strip(), str(city).strip() if pd.notna(city) else '')
    if key in geocode_cache:
        return geocode_cache[key]

    queries = [f'{key[0]}, {key[1]}' if key[1] else key[0]]
    if key[1]:
        queries.append(key[1])

    coords = (None, None)
    for q in queries:
        try:
            loc = geocode(q)
            if loc is not None:
                coords = (loc.latitude, loc.longitude)
                break
        except Exception:
            continue

    geocode_cache[key] = coords
    return coords

# Prefer stadiums with enough matches for a stable rate color
map_df = stadium_rates[stadium_rates['matches'] >= 3].copy()

lats, lons = [], []
for _, row in map_df.iterrows():
    lat, lon = lookup_coords(row['Stadium'], row['City'])
    lats.append(lat)
    lons.append(lon)

map_df['lat'] = lats
map_df['lon'] = lons
map_df = map_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)

print(f'Geocoded stadiums for maps: {len(map_df)} / {stadium_rates["matches"].ge(3).sum()} with >=3 matches')
map_df[['Stadium', 'City', 'matches', 'home_win_rate', 'away_win_rate', 'lat', 'lon']].head(10)

Geocoded stadiums for maps: 152 / 154 with >=3 matches


,Stadium,City,matches,home_win_rate,away_win_rate,lat,lon
0,Estadio Azteca,Mexico City,19,73.684211,10.526316,19.302979,-99.150470
1,Jalisco,Guadalajara,14,71.428571,21.428571,20.672037,-103.338396
2,Olympiastadion,Berlin West,14,42.857143,28.571429,38.332617,-75.361583
3,Nou Camp - Estadio Le�n,Leon,11,54.545455,27.272727,31.271513,-95.995338
4,Estadio do Maracana,Rio De Janeiro,10,60.000000,30.000000,-22.912162,-43.231186
5,Nacional,Santiago De Chile,10,90.000000,0.000000,-33.441082,-70.542740
6,Estadio Centenario,Montevideo,10,100.000000,0.000000,-34.894539,-56.152684
7,Estadio Nacional,Brasilia,10,70.000000,30.000000,-15.783750,-47.899029
8,Stade de France,Saint-Denis,9,44.444444,22.222222,48.924473,2.360133
9,Stade du Parc Lescure,Bordeaux,9,33.333333,11.111111,44.829308,-0.598264


In [19]:
# Two world maps: home win rate and away win rate by stadium
hover_cols = ['Stadium', 'City', 'matches']

fig_home = px.scatter_geo(
    map_df,
    lat='lat',
    lon='lon',
    color='home_win_rate',
    size='matches',
    hover_name='Stadium',
    hover_data={c: True for c in hover_cols} | {'home_win_rate': ':.1f', 'lat': False, 'lon': False},
    color_continuous_scale='RdYlGn',
    range_color=[0, 100],
    projection='natural earth',
    title='Home Team Win Rate by Stadium',
    size_max=28,
)
fig_home.update_layout(coloraxis_colorbar_title='Home win rate (%)', height=500)
fig_home.show()

fig_away = px.scatter_geo(
    map_df,
    lat='lat',
    lon='lon',
    color='away_win_rate',
    size='matches',
    hover_name='Stadium',
    hover_data={c: True for c in hover_cols} | {'away_win_rate': ':.1f', 'lat': False, 'lon': False},
    color_continuous_scale='RdYlGn',
    range_color=[0, 100],
    projection='natural earth',
    title='Away Team Win Rate by Stadium',
    size_max=28,
)
fig_away.update_layout(coloraxis_colorbar_title='Away win rate (%)', height=500)
# fig_away.show()